<a href="https://colab.research.google.com/github/cbonnin88/SoundStream/blob/main/Product_Metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import gdown as gd

In [3]:
url = 'https://drive.google.com/uc?id=1TkZ3_vwEH6zCZAa6QZWzhtFhKreHinbk'
gd.download(url,'soundstream_user_behavior_cleaned.csv',quiet=True)

'soundstream_user_behavior_cleaned.csv'

In [5]:
df_soundstream = pl.read_csv('soundstream_user_behavior_cleaned.csv')

In [6]:
display(df_soundstream.head())

user_id,country,age,signup_date,subscription_type,subscription_status,months_inactive,inactive_3_months_flag,ad_interaction,ad_conversion_to_subscription,music_suggestion_rating_1_to_5,avg_listening_hours_per_week,favorite_genre,most_liked_feature,desired_future_feature,primary_device,playlists_created,avg_skips_per_day
i64,str,i64,str,str,str,i64,i64,str,str,i64,f64,str,str,str,str,i64,i64
1,"""France""",25,"""2021-08-19""","""Premium Duo""","""Active""",0,0,"""No""","""No""",4,10.13,"""Bollywood""","""Radio""","""Concert Alerts""","""Tablet""",7,8
2,"""Indonesia""",20,"""2022-06-06""","""Premium Family""","""Active""",0,0,"""Yes""","""No""",5,11.63,"""Latin""","""Podcasts""","""Lyrics Translation""","""Mobile""",7,6
3,"""Italy""",53,"""2024-01-04""","""Premium Individual""","""Active""",0,0,"""Yes""","""Yes""",3,9.5,"""Bollywood""","""Lyrics""","""Better AI Recommendations""","""Desktop""",6,5
4,"""Italy""",48,"""2018-08-26""","""Premium Individual""","""Active""",1,0,"""No""","""No""",4,13.16,"""Electronic""","""Playlists""","""Social Listening""","""Smart Speaker""",11,8
5,"""Australia""",18,"""2020-05-29""","""Free""","""Active""",0,0,"""No""","""No""",4,12.7,"""Indie""","""Daily Mix""","""Lyrics Translation""","""Tablet""",10,11


# **1. COHORT ANALYSIS (Retention byu Signup Month)**

In [10]:
cohort_df = df_soundstream.with_columns(
    signup_month = pl.col('signup_date').str.to_datetime().dt.truncate('1mo')
).group_by('signup_month').agg([
    (pl.col('subscription_status') == 'Active').mean().alias('retention_rate'),
    pl.count('user_id').alias('cohort_size')
]).sort('signup_month')

display(cohort_df.head())

signup_month,retention_rate,cohort_size
datetime[μs],f64,u32
2018-01-01 00:00:00,0.854331,508
2018-02-01 00:00:00,0.82684,462
2018-03-01 00:00:00,0.830798,526
2018-04-01 00:00:00,0.836978,503
2018-05-01 00:00:00,0.818526,529


In [14]:
fig_cohort = px.line(
    cohort_df.to_pandas(),
    x='signup_month',
    y='retention_rate',
    title='Retention Health by Signup Cohort',
    labels={'retention_rate':'% Active','signup_month':'Join Month'},
    template = 'plotly_dark',
    markers= True
)

fig_cohort.show()

# **2. FORECASTING (User Acquisition Trends)**

In [15]:
# Logic: Simple 3-month moving average to forecast growth

growth_df = cohort_df.with_columns(
    moving_avg = pl.col('cohort_size').rolling_mean(window_size=3)
)

display(growth_df.head())

signup_month,retention_rate,cohort_size,moving_avg
datetime[μs],f64,u32,f64
2018-01-01 00:00:00,0.854331,508,null
2018-02-01 00:00:00,0.82684,462,null
2018-03-01 00:00:00,0.830798,526,498.666667
2018-04-01 00:00:00,0.836978,503,497.0
2018-05-01 00:00:00,0.818526,529,519.333333


In [17]:
fig_growth = px.bar(
    growth_df.to_pandas(),
    x='signup_month',
    y='cohort_size',
    title='User Acquisition & 3-Month Trendline',
    template='plotly_white',
    labels= {'signup_month':'Month Joined','cohort_size':'Cohort Size'}
)

fig_growth.add_traces(
    go.Scatter(x=growth_df['signup_month'], y=growth_df['moving_avg'], name='Growth Trend')
)

fig_growth.show()

# **3. FEEDBACK ANALYSIS (Feature REquest Popularity)**

In [18]:
# Logic: Count the frequency of 'desired_future_feature'

feedback_df = df_soundstream['desired_future_feature'].value_counts().sort('count',descending=True)

display(feedback_df.head())

desired_future_feature,count
str,u32
"""Mood-based Auto Playlists""",8419
"""Concert Alerts""",8403
"""Social Listening""",8369
"""Better AI Recommendations""",8335
"""Lyrics Translation""",8243


In [27]:
fig_feedback = px.bar(
    feedback_df.to_pandas().sort_values('count'),
    x='desired_future_feature',
    y='count',
    title='Market Demand: Top Requested Features',
    color='desired_future_feature',
    color_discrete_sequence=px.colors.sequential.Viridis_r,
    labels={'count':'Votes','desired_future_feature':'Desired Feature'}
)

fig_feedback.update_layout(showlegend=False)
fig_feedback.show()

# **4. CORRELATION MAPPING (Engagement Drivers)**

In [28]:
numeric_cols = ['age','music_suggestion_rating_1_to_5','avg_listening_hours_per_week','playlists_created','avg_skips_per_day']
corr_matrix = df_soundstream.select(numeric_cols).corr()

display(corr_matrix.head())

age,music_suggestion_rating_1_to_5,avg_listening_hours_per_week,playlists_created,avg_skips_per_day
f64,f64,f64,f64,f64
1.0,0.004236,-0.002308,0.001361,-0.004601
0.004236,1.0,0.000323,0.000023,0.005144
-0.002308,0.000323,1.0,0.001642,0.006923
0.001361,0.000023,0.001642,1.0,-0.006164
-0.004601,0.005144,0.006923,-0.006164,1.0


In [31]:
fig_corr = px.imshow(
    corr_matrix.to_numpy(),
    x=numeric_cols,
    y=numeric_cols,
    text_auto='.2f',
    title='Engagement Correlation Map',
    color_continuous_scale='Viridis_r',
    zmin=1,
    zmax=1
)

fig_corr.show()